# CycloPepper Dataset Audit (Phase 1)

**Purpose:** Inspect the publicly available CycloPepper data before model development. This notebook checks data structure, experimental labels, duplicate sequences, and cyclic rotation groups.

**Experimental labels:** Score = 1 means that a product signal at the expected molecular mass was detected by LC-MS under the reported conditions; Score = 0 means that such a signal was not observed. The scores do not directly measure cyclization yield or product purity.

**Data source:** Pan et al. (2026), *Nature Communications*. DOI: 10.1038/s41467-026-69441-w. [Original repository](https://github.com/Yongboxiao/Selection-of-Cyclization-Sites).

**External data:** The public `experiment.csv` contains 74 sequence records without a `Score` column. Experimental labels must be matched separately before calculating external predictive accuracy.

**Output:** `cyclopepper_dataset_audit.csv`. This phase does not train a predictive model.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

BASE = 'https://raw.githubusercontent.com/Yongboxiao/Selection-of-Cyclization-Sites/main/'
files = ['dataset.csv', 'experiment.csv', 'targets.csv', 'processed_sequences.csv']
data = {}
for filename in files:
    try:
        df = pd.read_csv(BASE + filename, encoding='utf-8-sig')
        data[filename] = df
        print(f'\n{filename}: shape={df.shape} columns={list(df.columns)}')
        display(df.head(5))
    except Exception as e:
        print(f'Could not load {filename}: {type(e).__name__}: {e}')



dataset.csv: shape=(306, 2) columns=['Sequence', 'Score']


,Sequence,Score
0,RG,1
1,HG,1
2,NG,1
3,QG,1
4,FW,1



experiment.csv: shape=(74, 1) columns=['Sequence']


,Sequence
0,MMVMPY
1,MVMPYM
2,VMPYMM
3,MPYMMV
4,PYMMVM



targets.csv: shape=(35, 3) columns=['Sequence', 'Therapeutic', 'Target']


,Sequence,Therapeutic,Target
0,GYTFTNY,Abciximab,CD41
1,YPGNSD,NaN,NaN
2,YDGYYVFAY,NaN,NaN
3,RASRDISNNLH,NaN,NaN
4,YASQSMS,NaN,NaN



processed_sequences.csv: shape=(295, 1) columns=['Sequence']


,Sequence
0,GYTFTNY
1,YTFTNYG
2,TFTNYGY
3,FTNYGYT
4,TNYGYTF


## 1. Inspect experimental labels

Check missing values, sequence lengths, duplicates, and the distribution of binary `Score` labels. These checks establish the data properties used in later classification experiments.


In [ ]:
df = data.get('dataset.csv')
if df is not None:
    print('Missing values:')
    display(df.isna().sum().to_frame('Missing count'))
    print('Data types:')
    print(df.dtypes)
    if 'Score' in df.columns:
        print('Score frequency:')
        display(df['Score'].value_counts(dropna=False).sort_index().to_frame('Count'))
        print('Score summary:')
        display(df['Score'].describe())
    if 'Sequence' in df.columns:
        seq = df['Sequence'].astype(str).str.strip()
        print('Duplicate exact sequences:', int(seq.duplicated().sum()))
        print('Lengths:')
        display(seq.str.len().value_counts().sort_index().to_frame('Count'))


Missing values:


,Missing count
Sequence,0
Score,0


Data types:
Sequence    object
Score        int64
dtype: object
Score frequency:


,Count
Score,
0,132
1,174


Score summary:


,Score
count,306.000000
mean,0.568627
std,0.496079
min,0.000000
25%,0.000000
50%,1.000000
75%,1.000000
max,1.000000


Duplicate exact sequences: 3
Lengths:


,Count
Sequence,
2,45
3,53
4,38
5,34
6,48
7,38
8,18
9,13
10,4


## 2. Inspect the external validation file

The public `experiment.csv` contains a `Sequence` column but no corresponding `Score` labels. The file alone is not sufficient for an external accuracy calculation.


In [ ]:
ext = data.get('experiment.csv')
if ext is not None:
    print('External file columns:', list(ext.columns))
    print('External rows:', len(ext))
    print('External has Score column:', 'Score' in ext.columns)
    display(ext.head(10))


External file columns: ['Sequence']
External rows: 74
External has Score column: False


,Sequence
0,MMVMPY
1,MVMPYM
2,VMPYMM
3,MPYMMV
4,PYMMVM
5,YMMVMP
6,WMLWEPF
7,MLWEPFW
8,LWEPFWM
9,WEPFWML


## 3. Identify cyclic rotation groups

For example, ABCD, BCDA, CDAB, and DABC belong to the same cyclic rotation group. Grouping related sequences supports leakage-aware model evaluation. Different labels for distinct linear precursors and conflicting labels for an identical sequence should be considered separately.


In [ ]:
def canonical_rotation(seq):
    seq = str(seq).strip().upper()
    if not seq or seq == 'NAN':
        return None
    return min(seq[i:] + seq[:i] for i in range(len(seq)))

if df is not None and 'Sequence' in df.columns:
    audit = df.copy()
    audit['rotation_group'] = audit['Sequence'].map(canonical_rotation)
    counts = audit['rotation_group'].value_counts(dropna=False)
    print('Unique rotation groups:', audit['rotation_group'].nunique())
    print('Groups with multiple rows:', int((counts > 1).sum()))
    display(counts.head(10).to_frame('Rows in group'))
    if 'Score' in audit.columns:
        grouped = audit.groupby('rotation_group')['Score'].nunique(dropna=False)
        print('Rotation groups with multiple distinct Score values:', int((grouped > 1).sum()))
        print('Note: interpret differences among distinct rotations separately from identical-sequence label conflicts.')


Unique rotation groups: 277
Groups with multiple rows: 21


,Rows in group
rotation_group,
DPGTFIL,4
ADGGTDD,4
FGIGPSY,3
EGMPNSF,3
AACAYRGF,3
DYGTKG,3
RS,2
GITPVIP,2
ACAYRGF,2


Rotation groups with multiple distinct Score values: 8
Note: if these scores differ, do not discard the site-specific differences.


## 4. Export the data audit

Save the checked records and rotation-group identifiers as a CSV file for subsequent analyses. Phase 2 independently loads the published dataset and performs model training.


In [ ]:
if df is not None:
    audit.to_csv('cyclopepper_dataset_audit.csv', index=False, encoding='utf-8-sig')
    try:
        from google.colab import files
        files.download('cyclopepper_dataset_audit.csv')
    except ImportError:
        print('Saved locally: cyclopepper_dataset_audit.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>